# Tutorial 5: Pure-Python Dev Loop

This notebook demonstrates the **spring-cleaning adapters** introduced in the Layer A refactor:

| New tool | What it does |
|----------|-------------|
| `run_pipeline()` | Full ingest + ML pipeline in one Python call — no Nextflow needed for dev |
| `ds.pano` accessor | Typed data contract on any `xr.Dataset`: `.validate()`, `.stamp()`, `.level`, `.kind` |
| `run_qc()` / QC report | Per-`(level, kind)` quality checks stamped alongside every L1 product |
| `FileCalibrationResolver` | Calibration params from a recipe YAML with provenance hash stamped in attrs |
| `@qc_check` decorator | Register your own QC checks — no boilerplate |

**Prerequisites**: `uv sync` in the repo root; the bundled `tests/data/obs_TEST.pffd` is used for all live cells.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from panoseti_analysis.paths import REPO_ROOT

# Bundled test observation directory
OBS_DIR = REPO_ROOT / "tests" / "data" / "obs_TEST.pffd"
OUT_DIR = REPO_ROOT / "notebooks" / "tutorials" / "work" / "dev_loop_demo"

print(f"OBS_DIR exists: {OBS_DIR.exists()}")
print(f"OUT_DIR      : {OUT_DIR}")

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

---
## §1  `run_pipeline()` — the pure-Python dev driver

`run_pipeline` mirrors the Nextflow `ingest.nf + ml.nf` DAG as a flat sequence:

```
PFF → L0 → (HK) → L1 (ph/img calibration) → manifests → L2 (cloud, img only)
```

It is **not** a second orchestrator — no DAG engine, no resume, no SLURM.  
Use Nextflow for production.  Use `run_pipeline` for:
- notebook R&D on a single run
- integration tests without Nextflow installed
- quick end-to-end smoke tests on dev machines

In [ ]:
from panoseti_analysis.adapters.recipe_driver import run_pipeline

if OUT_DIR.exists():
    import shutil

    shutil.rmtree(OUT_DIR)  # start fresh each run

outputs = run_pipeline(OBS_DIR, OUT_DIR)

print(f"L0 stores  : {len(outputs.l0_stores)}")
print(f"HK stores  : {len(outputs.hk_stores)}")
print(f"L1 stores  : {len(outputs.l1_stores)}")
print(f"L2 stores  : {len(outputs.l2_stores)} (no model → skipped)")
print()
if outputs.l0_manifest:
    print(f"L0 manifest: {outputs.l0_manifest.run_id}  ({len(outputs.l0_manifest.stores)} stores)")
if outputs.l1_manifest:
    print(f"L1 manifest: {outputs.l1_manifest.run_id}  ({len(outputs.l1_manifest.stores)} stores)")

In [ ]:
# RunOutputs carries typed StoreLineage records — no filesystem string-parsing
for rec in outputs.l1_stores:
    print(
        f"  {rec.store:<50}  kind={rec.kind}  frames={rec.n_frames}  qc_stamped={'qc' in (rec.calibration_params or {})}"
    )

In [ ]:
# With a model, L2 classification runs automatically:
#   from panoseti_analysis.paths import MODELS
#   outputs = run_pipeline(OBS_DIR, OUT_DIR, model_path=MODELS / "cloud_detector_v1.pt")
# Skip here to avoid the model download in CI.
print("(model_path not supplied → L2 step skipped)")

---
## §2  `ds.pano` — the typed data contract

Every `xr.Dataset` loaded from a PANOSETI Zarr store automatically has a `.pano` accessor.  
The accessor is registered as a side-effect of importing `panoseti_analysis.config` — nothing extra to do.

| Method / property | Behaviour |
|---|-----------|
| `ds.pano.level` | `ds.attrs["data_level"]` (or `None`) |
| `ds.pano.kind` | Inferred from `ds.attrs["data_product"]` — `"img"` or `"ph"` |
| `ds.pano.validate(level, kind)` | Raises `ValueError` naming exactly what is missing; returns `ds` (chainable) |
| `ds.pano.stamp(data_level, ...)` | Returns a **new** Dataset with canonical attrs merged; sole writer of `data_level` / `storage_version` / `calibration` / `qc` keys |

In [ ]:
from panoseti_analysis.io.stores import open_store

# Load one of the L1 stores produced in §1
l1_rec = outputs.l1_stores[0]
l1_path = OUT_DIR / "L1" / l1_rec.store
ds_l1 = open_store(l1_path)

print(f"ds.pano.level : {ds_l1.pano.level!r}")
print(f"ds.pano.kind  : {ds_l1.pano.kind!r}")
print(f"data_vars     : {list(ds_l1.data_vars)}")

In [ ]:
# validate() is a pre-flight check: raises immediately if the contract is broken
kind = ds_l1.pano.kind
ds_l1.pano.validate(level="L1", kind=kind)
print(f"✓ Dataset is a valid L1/{kind} store")

In [ ]:
# validate() raises a precise error on a contract violation

bad_ds = xr.Dataset({"median_subtracted": (["T"], [1.0])}, attrs={"data_product": "img16"})
try:
    bad_ds.pano.validate(level="L1", kind="img")
except ValueError as exc:
    print(f"ValueError: {exc}")

In [ ]:
# stamp() is the sole writer of canonical attrs — never mutate ds.attrs directly.
# It returns a NEW dataset (non-mutating).
stamped = ds_l1.pano.stamp(
    data_level="L1",
    extra_attrs={"custom_key": "demo"},
)
print(f"Original  data_level : {ds_l1.attrs.get('data_level')}")
print(f"Stamped   data_level : {stamped.attrs.get('data_level')}")
print(f"Custom key preserved : {stamped.attrs.get('custom_key')}")
print(f"Original unchanged   : {'custom_key' not in ds_l1.attrs}")

### SchemaSpec registry

The contract for each `(level, kind)` lives in `config/schema.py`.  
You can inspect or extend it:

In [ ]:
from panoseti_analysis.config.schema import SchemaSpec, get_schema, known_schemas, register_schema

print("Registered schemas:", known_schemas())
print()
spec = get_schema("L1", "img")
print(f"L1/img required vars  : {spec.required_vars}")
print(f"L1/img required attrs : {spec.required_attrs}")

In [ ]:
# Registering a new schema for a custom product level:
register_schema(SchemaSpec("L3", "coincidence", required_vars=("coincidence_score", "unix_t_ns")))
print("After registration:", known_schemas())

---
## §3  QC reports — stamped alongside every L1 product

Every L1 store produced by `run_calibrate` (or `run_pipeline`) carries a `qc` attrs block.  
The same `run_qc()` function works anywhere in pure Python — no I/O needed.

In [ ]:
# The QC report is already in attrs — zero extra cost
qc = ds_l1.attrs.get("qc", {})
print(f"isgood  : {qc.get('isgood')}")
print(f"metrics : {qc.get('metrics')}")
print()
for check in qc.get("checks", []):
    status = "✓" if check["passed"] else "✗"
    print(
        f"  {status} {check['key']:<25} value={check['value']:.4f}  threshold={check['threshold']:.4f}"
    )

In [ ]:
# run_qc() works on any Dataset — useful in ad-hoc exploration
from panoseti_analysis.algorithms.qc import registered_checks, run_qc

kind = ds_l1.pano.kind
print(f"Registered checks for L1/{kind}:", registered_checks("L1", kind))

report = run_qc(ds_l1, level="L1", kind=kind)
print(f"\nrun_qc → isgood={report.isgood}")
for c in report.checks:
    bar = "█" * int(c.value * 40) if c.key != "frame_rate_hz" else "(hz)"
    print(f"  {c.key:<25} {c.value:8.4f}  {'PASS' if c.passed else 'FAIL'}")

### Adding a custom QC check

Decorating a function with `@qc_check` registers it immediately.  Any subsequent call to `run_qc(ds, level=..., kind=...)` will include it.

In [ ]:
from panoseti_analysis.algorithms.qc import QCCheckResult, qc_check


# Example: flag L1 img stores whose time-axis has gaps > 5 s
@qc_check("L1", "img", "max_gap_s")
def check_max_time_gap(ds: xr.Dataset) -> QCCheckResult:
    t = np.asarray(ds["unix_t_ns"].values)
    max_gap_s = float(np.max(np.diff(t))) / 1e9 if t.size > 1 else 0.0
    return QCCheckResult(
        key="max_gap_s",
        value=max_gap_s,
        threshold=5.0,  # flag if any gap > 5 s
        passed=max_gap_s <= 5.0,
    )


report2 = run_qc(ds_l1, level="L1", kind="img")
max_gap = next(c for c in report2.checks if c.key == "max_gap_s")
print(f"max_gap_s check: value={max_gap.value:.3f} s  passed={max_gap.passed}")

---
## §4  Calibration recipes — `FileCalibrationResolver`

Passing a recipe YAML to `run_calibrate` (or `run_pipeline`) stamps a `CalibrationResolution` record into `ds.attrs["calibration"]["resolution"]`.  This makes every L1 store traceable to an exact calibration configuration by hash.

In [ ]:
# Write a minimal calibration recipe
recipe_path = OUT_DIR / "calib_demo.yml"
recipe_path.parent.mkdir(parents=True, exist_ok=True)
recipe_path.write_text("""
name: calib_demo
calibration:
  img:
    frame_stride: 100
    block_size: 8
    adc_to_pe: 2.0
  ph:
    sigma_threshold: 4.0
    baseline_offset: 750
    frame_stride: 100
""")
print(f"Recipe written: {recipe_path}")

In [ ]:
# The FileCalibrationResolver API (what run_calibrate uses internally)
from panoseti_analysis.io.calibration_source import FileCalibrationResolver

resolver = FileCalibrationResolver(recipe_path)
print(f"Recipe name : {resolver.recipe_name}")
print(f"Recipe hash : {resolver.recipe_hash}")

for cal_type in ("img", "ph"):
    res = resolver.resolve(cal_type, context={})
    print(f"\n{cal_type} params:")
    for k, v in res.params.items():
        print(f"  {k:<20} {v}")

In [ ]:
# Run the pipeline with the calibration recipe
out2_dir = OUT_DIR / "with_recipe"
if out2_dir.exists():
    import shutil

    shutil.rmtree(out2_dir)

outputs2 = run_pipeline(OBS_DIR, out2_dir, calib_recipe=recipe_path)

# Inspect the resolution stamped into L1 attrs
for rec in outputs2.l1_stores:
    l1_path2 = out2_dir / "L1" / rec.store
    ds2 = open_store(l1_path2)
    cal = ds2.attrs.get("calibration", {})
    resolution = cal.get("resolution", None)
    if resolution:
        print(f"{rec.store}")
        print(f"  recipe hash   : {resolution['source_hash'][:30]}...")
        print(f"  relevance     : {resolution['relevance']}")
        print(f"  adc_to_pe     : {resolution['params'].get('adc_to_pe')}")  # should be 2.0
        print()

---
## §5  Putting it all together — dev workflow

A typical R&D loop with the new adapters:

```python
# 1. Run the full pipeline (ingest + optional ML) in one call
outputs = run_pipeline(obs_dir, out_dir, model_path=MODEL, calib_recipe=RECIPE)

# 2. Load a store — the accessor is ready immediately
ds = open_store(out_dir / "L1" / outputs.l1_stores[0].store)

# 3. Validate the contract before doing science
ds.pano.validate(level="L1", kind=ds.pano.kind)

# 4. Read the QC report — already stamped by the pipeline
qc = ds.attrs["qc"]
if not qc["isgood"]:
    failed = [c["key"] for c in qc["checks"] if not c["passed"]]
    print(f"QC failures: {failed}")

# 5. Check calibration provenance
cal = ds.attrs["calibration"]
print(f"Recipe hash: {cal['resolution']['source_hash']}")
print(f"adc_to_pe  : {cal['resolution']['params']['adc_to_pe']}")

# 6. Do science (the contract guarantees the vars you need are present)
median_subtracted = ds["median_subtracted"]
```

In [ ]:
# Live example of the above pattern
ds = open_store(out2_dir / "L1" / outputs2.l1_stores[0].store)
ds.pano.validate(level="L1", kind=ds.pano.kind)

qc = ds.attrs["qc"]
cal = ds.attrs["calibration"]

print(f"Store       : {outputs2.l1_stores[0].store}")
print(f"Level/kind  : {ds.pano.level} / {ds.pano.kind}")
print(f"QC isgood   : {qc['isgood']}")
print(f"Recipe hash : {cal.get('resolution', {}).get('source_hash', 'no recipe')[:30]}...")
print(f"adc_to_pe   : {cal.get('resolution', {}).get('params', {}).get('adc_to_pe', 'default')}")
print()
print(f"median_subtracted shape: {ds['median_subtracted'].shape}")

---
## §6  Quick reference

### New CLIs

| Command | What it does |
|---------|-------------|
| `pa-run <obs_dir> <out_dir>` | Full pipeline (Nextflow-free dev mode) |
| `pa-calibrate … --recipe calib.yml` | Calibrate with provenance from recipe |
| `pa-calibrate … --qc-out store.qc.json` | Write QC sidecar alongside L1 |

### Key import paths

```python
from panoseti_analysis.adapters.recipe_driver import run_pipeline, RunOutputs
from panoseti_analysis.algorithms.qc       import run_qc, qc_check, registered_checks
from panoseti_analysis.config.schema       import get_schema, register_schema, SchemaSpec
from panoseti_analysis.config.calibration  import CalibrationResolver, CalibrationResolution
from panoseti_analysis.io.calibration_source import FileCalibrationResolver
from panoseti_analysis.io.qc               import stamp_qc, write_qc_sidecar
```

### Design rules (see CLAUDE.md §Design principles)

- **`ds.pano.stamp()`** is the sole writer of `data_level`, `storage_version`, `calibration`, `qc` attrs — never set them directly.
- **`ds.pano.validate()`** at every adapter boundary — never let a missing variable crash downstream.
- **`run_pipeline()`** is for notebooks/dev — **Nextflow** is the production orchestrator.